# ConnectaTel Customer Usage & Segmentation Analysis

> **Project context:** This is a guided portfolio project completed as part of the **TripleTen Data Analyst Bootcamp**. TripleTen provided the case framework and datasets; the data cleaning, feature engineering, statistical analysis, visualizations, customer segmentation, and business interpretation were completed as part of my project work.

## Business Objective

Analyze ConnectaTel customer behavior to identify data-quality issues, understand usage patterns across plan types, build customer-level usage metrics, detect outliers, segment the customer base, and translate the findings into commercial recommendations.

## Tools & Methods

- Python
- Pandas and NumPy
- Seaborn and Matplotlib
- Descriptive statistics
- IQR outlier analysis
- Customer segmentation
- Business-oriented interpretation

> Plan labels such as `Basico` and `Premium` are preserved as they appear in the source data.


---
## 1. Data Loading and Initial Exploration

The analysis begins by loading the three source datasets and reviewing their structure before any cleaning or transformation is applied.


In [ ]:
# Import libraries
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Load files
plans = pd.read_csv('/datasets/plans.csv')
users = pd.read_csv('/datasets/users_latam.csv')
usage = pd.read_csv('/datasets/usage.csv')

In [ ]:
# Display the first 5 rows of plans
plans.head(5)

In [ ]:
# Display the first 5 rows of users
users.head(5)

In [ ]:
# Display the first 5 rows of usage
usage.head(5)

### 1.1 Dataset Structure Review

Rows, columns, data types, and missing values are reviewed to establish the initial condition of each dataset.


In [ ]:
# Review the number of rows and columns in each dataset
print("plans", plans.shape)
print("users", users.shape)
print("usage", usage.shape)

In [ ]:
# Inspect plans with .info()
plans.info()

In [ ]:
# Inspect users with .info()
users.info()

In [ ]:
# Inspect usage with .info()
usage.info()

---
## 2. Data Quality Assessment

### 2.1 Missing Values

Missing-value counts and percentages are reviewed first. The relationship between missing `duration` / `length` values and interaction `type` is then examined to determine whether those gaps are structurally expected.


In [ ]:
# Count missing values in users
print(users.isna().sum())
print(users.isna().mean() * 100)

In [ ]:
# Count missing values in usage
print(usage.isna().sum())
print(usage.isna().mean() * 100)

In [ ]:
# Review the number of records by usage type
print(usage['type'].value_counts())

In [ ]:
# Calculate the proportion of missing values in duration and length by usage type
print(
    usage.groupby('type')[['duration', 'length']]
         .apply(lambda x: x.isna().mean())
)

In [ ]:
# Display sample records to review the relationship between type, duration, and length
print(
    usage[['type', 'duration', 'length']]
    .head(20)
)

In [ ]:
# Calls with a non-null length value
print(
    usage[(usage['type'] == 'call') & (usage['length'].notna())]
)

In [ ]:
# Text messages with a non-null duration value
print(
    usage[(usage['type'] == 'text') & (usage['duration'].notna())]
)

### 2.2 Invalid Values and Sentinels

Numerical summaries and categorical frequencies are used to identify invalid values, sentinels, and suspicious records that may require cleaning or further review.


In [ ]:
# Explore numerical columns in users
users.describe()

- The `user_id` column contains 4,000 records, with consecutive values from 10,000 to 13,999.
- The `age` column also contains 4,000 records. The median is 47 years, 25% of the values are less than or equal to 32 years, and 75% are less than or equal to 63 years. A minimum value of -999 is observed, which is a possible sentinel value and distorts the variable's mean.


In [ ]:
# Explore numerical columns in usage
usage.describe()

- The `duration` and `length` columns contain 17,924 and 22,104 recorded values, respectively. `duration` has an average of 5.2 and is mainly associated with `call` records, while `length` has an average of 52.1 and is mainly associated with `text` records. Because no data dictionary is available, the unit of measurement for `length` cannot be determined with certainty.
- Maximum values of 120 in `duration` and 1490 in `length` are observed, considerably above their respective quartiles, so they should be investigated as possible outliers or sentinels.


In [ ]:
# Explore categorical columns in users
user_columns = ['city', 'plan']

print("Number of unique values")
print(users[user_columns].nunique())

# Absolute and relative frequency of city
print(users['city'].value_counts(dropna=False))
print(users['city'].value_counts(dropna=False, normalize=True) * 100)

# Absolute and relative frequency of plan
print(users['plan'].value_counts())
print(users['plan'].value_counts(normalize=True) * 100)

- The `city` column contains 7 distinct non-null values: six cities (Bogotá, CDMX, Medellín, GDL, Cali, and MTY) and one suspicious value (`"?"`). In addition, there are 469 missing values, representing 11.7% of the 4,000 records.
- Bogotá has the highest number of records, with 808 (20.2%), followed by CDMX with 730 (18.3%) and Medellín with 616 (15.4%). MTY has the fewest records, with 407 (10.2%).
- The value `"?"` appears in 96 records (2.4%) and is identified as a possible sentinel because it does not correspond to a valid city. It is recommended to treat it later as a missing value.
- The `plan` column contains 2 unique values: `Basico` and `Premium`. `Basico` is the most frequent plan, with 64.9% of the records, while `Premium` represents 35.1%.


In [ ]:
# Explore the categorical column in usage
print("Absolute frequency by category")
print(usage['type'].value_counts())

print("Relative frequency by category")
print(usage['type'].value_counts(normalize=True) *100)

- The `type` column contains two categories: `text` and `call`. No missing values were identified in this column.
- `text` contains 22,092 records, equivalent to 55.2%, while `call` contains 17,908 records, equivalent to 44.8%.
- No invalid values or sentinels are observed in the `type` column, since it contains only the expected categories `text` and `call`.


**Invalid Values or Sentinels**

- Invalid values or sentinels were identified in `age`, where -999 appears, and in `city`, where `"?"` appears. Suspicious values were also observed in `duration` and `length`: some `text` records have `duration = 120`, and some `call` records have `length = 1490`, even though these variables normally do not correspond to those usage types.
- It is recommended to convert -999 and `"?"` into missing values.
- The inconsistent `duration` and `length` values should also be treated as missing after confirming that they correspond to records where those variables do not apply.


### 2.3 Date Validation

Date fields are converted to datetime format and reviewed for years outside the expected reporting period. The dataset contains valid records through 2024.


In [ ]:
# Convert the `reg_date` column in users to datetime
users['reg_date'] = pd.to_datetime(users['reg_date'])

In [ ]:
# Convert the `date` column in usage to datetime
usage['date'] = pd.to_datetime(usage['date']) 

In [ ]:
# Review the years present in `reg_date` in users
users['reg_date'].dt.year.unique()

In `reg_date`, the years present are 2022, 2023, 2024, and 2026. The year 2026 falls outside the expected range because the database records are limited to 2024.

In [ ]:
# Review the years present in `date` in usage
usage['date'].dt.year.unique()

In `date`, the only valid year present is 2024. The `NaN` values correspond to previously detected missing dates rather than out-of-range years.

In [ ]:
# Count records by year in reg_date
users['reg_date'].dt.year.value_counts(dropna=False)

- A total of 40 records with `reg_date` in 2026 were identified, so they should be treated as out-of-range dates during the cleaning stage.

---
## 3. Data Cleaning

### 3.1 Sentinels and Out-of-Range Dates

Cleaning rules are applied to the issues identified during the quality assessment:

- replace the `age` sentinel `-999` with the median age,
- replace `"?"` in `city` with a missing value,
- mark registration dates beyond 2024 as missing.


In [ ]:
# Replace -999 with the median age
age_median = users['age'].median()
users['age'] = users['age'].replace(-999, age_median)

# Verify changes
users['age'].describe()

In [ ]:
# Replace ? with NA in city
users['city'] = users['city'].replace('?', pd.NA)

# Verify changes
print(users['city'].describe())
print((users['city'] == '?').any())

In [ ]:
# Mark future reg_date values as NA
users.loc[users['reg_date'].dt.year >2024, 'reg_date'] = pd.NaT

# Verify changes
print(users['reg_date'].describe())
print((users['reg_date'].dt.year >2024).any())

### 3.2 Structurally Expected Missing Values

Missingness in `duration` and `length` is evaluated by interaction type. Values that correspond to fields that do not apply to a given interaction are retained as missing rather than imputed.


In [ ]:

# MAR (Missing At Random) check for duration in usage


print(
    usage.groupby('type')['duration']
         .apply(lambda x: x.isna().mean() * 100)
)


In [ ]:
# MAR (Missing At Random) check for length in usage

print(
    usage.groupby('type')['length']
         .apply(lambda x: x.isna().mean() * 100)
)

Missing values in `duration` and `length` are explained by the interaction `type`: `length` is generally not applicable to `call` records, while `duration` is generally not applicable to `text` records. These values are therefore retained as missing rather than imputed.


---
## 4. Customer-Level Feature Engineering

### 4.1 Usage Aggregation

Interaction-level records are aggregated by `user_id` to create three customer-level usage metrics:

- `msg_cnt`: total messages,
- `call_cnt`: total calls,
- `call_min`: total accumulated call minutes.

The aggregated usage table is then merged with the customer dataset.


In [ ]:
# Helper columns

usage["is_text"] = (usage["type"] == "text").astype(int) # count total messages
usage["is_call"] = (usage["type"] == "call").astype(int) # count total calls
usage["is_duration"] = np.where(usage["type"] == "call", usage['duration'], 0) # sum total call minutes


# Aggregate information by user
usage_agg = usage.groupby('user_id').agg({
                        'is_text' : sum,
                        'is_call' : sum,
                        'is_duration' : sum, 
    }).reset_index()

# Review result
usage_agg.head(3)



In [ ]:
# Rename columns
usage_agg = usage_agg.rename(columns={
            'is_text' : 'msg_cnt',
            'is_call' : 'call_cnt',
            'is_duration' : 'call_min'
})

# Review result
usage_agg.head(3)

In [ ]:
# Merge the aggregated table with the users dataset

user_profile = users.merge(usage_agg, on='user_id', how='left')
user_profile.head(5)


### 4.2 Customer-Level Statistical Summary

Descriptive statistics are reviewed for customer demographics and usage metrics. Plan composition is also measured to establish the relative size of the `Basico` and `Premium` customer bases.


In [ ]:
# Statistical summary of numerical columns
user_profile.describe()

In [ ]:
# Replace missing usage metrics with 0
user_profile[['msg_cnt', 'call_cnt', 'call_min']] = (
    user_profile[['msg_cnt', 'call_cnt', 'call_min']].fillna(0)
)

# Verify changes
user_profile.describe()

In [ ]:
# Percentage distribution by plan type
(user_profile['plan'].value_counts(normalize=True) *100)

---
## 5. Exploratory Distribution Analysis

### 5.1 Customer and Usage Distributions

Histograms are used to examine age, message count, call count, and accumulated call minutes by plan type. The focus is on distribution shape, concentration, and visible similarities or differences between `Basico` and `Premium`.


In [ ]:
# Histogram of customer age

sns.histplot(
    data=user_profile,
    x='age',
    hue='plan',
    palette=['skyblue', 'green'],
    binwidth=5
)
plt.title("User Distribution by Age and Plan Type")
plt.xlabel("Age")
plt.ylabel("Users")
plt.xticks(range(20, 81, 5))
plt.show()


💡 **Insights:**
- The age distribution is relatively uniform, with no clear skew toward older or younger customers, although there is a higher concentration in the 45–50 age range. No evident differences are observed in the age distribution between the `Basico` and `Premium` plans. The higher frequencies in `Basico` are consistent with its larger share of the customer base, close to 65%.


In [ ]:
# Histogram of message count

sns.histplot(
    data=user_profile,
    x='msg_cnt',
    hue='plan',
    palette=['skyblue', 'green'],
    multiple='dodge',
    discrete=True
)
plt.title("User Distribution by Number of Messages and Plan Type")
plt.xlabel("Message Count")
plt.ylabel("Users")
plt.xticks(range(0, 18, 1))
plt.show()

💡 **Insights:**
- The highest concentration of customers is between 4 and 6 messages sent.
- The distribution is right-skewed, as the number of customers decreases as the number of messages increases, up to a maximum of 17.
- The `Basico` and `Premium` plans show a similar distribution shape, although `Basico` has higher frequencies because it contains a larger share of customers.


In [ ]:
# Histogram of call count
sns.histplot(
    data=user_profile,
    x='call_cnt',
    hue='plan',
    palette=['skyblue', 'green'],
    multiple='dodge'
)

plt.title("User Distribution by Number of Calls and Plan Type")
plt.xlabel("Calls Count")
plt.ylabel("Users")
plt.xticks(range(0, 16, 1))
plt.show()

💡 **Insights:**
- The highest concentration of customers is between 3 and 5 calls.
- The distribution is right-skewed, as the number of customers decreases as the number of calls increases, up to a maximum of 15.
- The `Basico` and `Premium` plans show a similar distribution shape. `Basico` has higher frequencies mainly because it contains a larger share of customers.


In [ ]:
# Histogram of total call minutes
sns.histplot(
    data=user_profile,
    x='call_min',
    hue='plan',
    palette=['skyblue', 'green'],
    binwidth=10
)
plt.title("User Distribution by Total Call Minutes per User and Plan Type")
plt.xlabel("Total Call Minutes per User")
plt.ylabel("Users")
plt.xticks(range(0, 160, 10))
plt.show()

💡 **Insights:**
- 75% of customers accumulate 31.2 total call minutes or less.
- The distribution is right-skewed: most customers accumulate relatively few total call minutes, and the frequency decreases as accumulated time increases, up to a maximum of 155.7 minutes per customer.
- The `Basico` and `Premium` plans show a similar distribution shape. `Basico` has higher frequencies mainly because it contains a larger share of customers.


### 5.2 Outlier Analysis

Boxplots are used to identify extreme values in `age`, `msg_cnt`, `call_cnt`, and `call_min`. For variables with upper-end outliers, IQR thresholds are calculated and compared with observed maxima before deciding whether the values should be retained.


In [ ]:
# Visualize distributions using boxplots
numeric_columns = ['age', 'msg_cnt', 'call_cnt', 'call_min']

for col in numeric_columns:
    sns.boxplot(
        data=user_profile,
        x=col,
    )
    plt.title(f'Boxplot: {col}')
    plt.show()

💡 **Insights:**
- `age`: the boxplot does not show outliers.
- `msg_cnt`: the boxplot shows upper-end outliers.
- `call_cnt`: the boxplot shows upper-end outliers.
- `call_min`: the boxplot shows upper-end outliers.


In [ ]:

# Calculate upper limits using the IQR method

limit_columns = ['msg_cnt', 'call_cnt', 'call_min']

for col in limit_columns:
    Q1 = user_profile[col].quantile(.25)
    Q3 = user_profile[col].quantile(.75)
    IQR = Q3 - Q1 
    upper_limit = Q3 + (1.5 * IQR)
    print(f'{col}: {upper_limit:.1f}')


In [ ]:
# Review upper limits and maximum values to decide whether to retain outliers
user_profile[limit_columns].describe()

💡 **Insights:**

- `msg_cnt`: the upper limit calculated using IQR is 11.5 messages, so values of 12 or more are considered outliers. The maximum is 17 messages per customer, which is plausible, so these values should be retained.
- `call_cnt`: the upper limit is 10.5 calls, so values of 11 or more are considered outliers. The maximum is 15 calls per customer, which is plausible, so these values should be retained.
- `call_min`: the upper limit is 61.3 accumulated call minutes per customer, and the maximum is 155.7 minutes. Although some values are considerably high, they remain plausible for customers with higher activity, so they should be retained. If necessary, only the visualization should be adjusted to better display the distribution.


---
## 6. Customer Segmentation

### 6.1 Segmentation by Usage Level

Customers are classified according to call and message activity:

- `Low Usage`: fewer than 5 calls and fewer than 5 messages,
- `Medium Usage`: customers not classified as Low Usage who have fewer than 10 calls and fewer than 10 messages,
- `High Usage`: all remaining customers.


In [ ]:
# Create the usage_group column
user_profile['usage_group'] = np.where(
    (user_profile['call_cnt'] < 5) & (user_profile['msg_cnt'] < 5), 'Low Usage',
    np.where(
        (user_profile['call_cnt'] < 10) & (user_profile['msg_cnt'] < 10), 'Medium Usage', 'High Usage'),
)

In [ ]:
# Verify changes
user_profile.head()

In [ ]:
# Customers by usage segment
user_profile['usage_group'].value_counts()

### 6.2 Segmentation by Age

Customers are grouped into three age segments:

- `Young`: under 30,
- `Adult`: 30–59,
- `Older Adult`: 60 or older.


In [ ]:
# Create the age_group column
user_profile['age_group'] = np.where(
    user_profile['age'] < 30, 'Young',
    np.where(user_profile['age'] < 60, 'Adult', 'Older Adult')
)

In [ ]:
# Verify changes
print("Overview:")
print(user_profile.head())

print()

print("Segmentation:")
print(user_profile[['age', 'msg_cnt', 'call_cnt', 'usage_group', 'age_group']].head())

In [ ]:
# Customers by age group
user_profile['age_group'].value_counts()

### 6.3 Segment Distribution

The resulting usage and age segments are visualized and measured as percentages of the customer base.


In [ ]:
# Visualize usage segments
sns.countplot(
    data=user_profile,
    x='usage_group',
    order=['Low Usage', 'Medium Usage', 'High Usage']
)
plt.title('Customer Distribution by Usage Level')
plt.xlabel('Usage Level')
plt.ylabel('Number of Customers')
plt.show()

In [ ]:
# Percentage of customers by usage level
user_profile['usage_group'].value_counts(normalize=True) * 100

In [ ]:
# Visualize age segments
sns.countplot(
    data=user_profile,
    x='age_group',
    order=['Young', 'Adult', 'Older Adult']
)
plt.title('Customer Distribution by Age Group')
plt.xlabel('Age Group')
plt.ylabel('Number of Customers')
plt.show()

In [ ]:
# Percentage of customers by age group
user_profile['age_group'].value_counts(normalize=True) * 100

---
## 7. Business Findings for Stakeholders

### Data Quality Findings

- The `users` table contained missing values in `city` and `churn_date`, representing approximately **11.7%** and **88.4%** of records, respectively.
- `city` also contained the sentinel value `"?"` in **96 records (2.4%)**, which was replaced with missing values.
- `age` contained the sentinel value `-999`, which was replaced with the median age.
- `reg_date` contained **40 records (1.0%)** from 2026, outside the expected reporting period through 2024; these dates were marked as missing.
- The `usage` table contained missing values in `date`, `duration`, and `length`, representing approximately **0.1%**, **55.2%**, and **44.7%** of records.
- Missing `duration` and `length` values are largely explained by interaction type and were retained when the field did not apply to that interaction.
- Suspicious cross-field values were also observed, including `text` records with `duration = 120` and `call` records with `length = 1490`.
- One customer had no records in `usage`. A left merge preserved all **4,000 customers**, and that customer's usage metrics were set to zero.

### Customer Segments

**By usage level**

- `Medium Usage`: **73.6%**
- `Low Usage`: **19.5%**
- `High Usage`: **7.0%**

**By age**

- `Adult` (30–59): **50.5%**
- `Older Adult` (60+): **30.6%**
- `Young` (<30): **19.0%**

The customer base is therefore concentrated mainly among customers aged 30 or older and within the `Medium Usage` segment.

### Plan Distribution

- `Basico`: approximately **64.9%** of customers.
- `Premium`: approximately **35.1%** of customers.
- Age, message, call, and accumulated-minute distributions show broadly similar shapes across both plans.
- The higher absolute frequencies observed for `Basico` are consistent with its larger customer base.

The size of the `Basico` base makes it especially relevant for retention, increased engagement, and selective migration to higher-value offers. The smaller `Premium` base represents an opportunity to expand adoption through differentiated benefits and promotions.

### Outlier Findings

Upper-end outliers were identified in the main usage variables:

- `msg_cnt`: IQR upper threshold **11.5**; values of 12+ are outliers; maximum **17**.
- `call_cnt`: IQR upper threshold **10.5**; values of 11+ are outliers; maximum **15**.
- `call_min`: IQR upper threshold approximately **61.3 accumulated minutes**; maximum **155.7**.

These extreme values remain plausible for highly active customers and are retained. Commercially, such customers may be relevant for loyalty programs and offers aimed at higher usage levels.

### Commercial Recommendations

- Prioritize retention and increased engagement among `Medium Usage` customers, who represent nearly three-quarters of the customer base.
- Capitalize on the large `Basico` customer base through retention initiatives, greater service usage, and selective migration to higher-value offers.
- Strengthen `Premium` adoption through differentiated benefits, promotions, and a clearer value proposition.
- Develop loyalty initiatives for `High Usage` customers.
- Address `Low Usage` customers with engagement initiatives designed to reduce potential churn.
- Maintain a competitive value proposition for the `Adult` segment, which represents roughly half of the customer base.
- Strengthen the offer for `Young` customers to increase their representation within the customer base.


---
## 8. Executive Summary

ConnectaTel's customer base is predominantly **adult**, concentrated in the **Medium Usage** segment, and primarily subscribed to the **Basico** plan.

The strongest near-term commercial opportunity lies in the large `Medium Usage` and `Basico` customer bases. Their size supports prioritizing retention, deeper engagement, and selective migration to higher-value offers. `Premium`, which represents approximately one-third of customers, offers additional growth potential through differentiated benefits and targeted promotions.

`High Usage` customers form a smaller but strategically relevant segment for loyalty initiatives, while `Low Usage` customers warrant engagement actions aimed at strengthening service use and reducing potential churn. The lower representation of `Young` customers also suggests room to strengthen the value proposition for this segment.

### Recommended Priorities

1. **Retain and deepen engagement** among `Medium Usage` customers.
2. **Monetize the scale of `Basico`** through usage growth and selective upgrades.
3. **Grow `Premium` adoption** through differentiated benefits and targeted promotions.
4. **Protect high-value engagement** with loyalty initiatives for `High Usage` customers.
5. **Reduce disengagement risk** among `Low Usage` customers.
6. **Strengthen the proposition for younger customers** to increase their participation in the customer base.
